# 🔬 Arm 2 (Priority 2 — Distillation): Contrastive Thought-Template SFT / DPO
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada Soliman  
**Literature:** SuperCorrect (ICLR 2025 / NeurIPS 2024) · ReCode (2025) · Contrastive CoT (Chia et al., 2023)  

---

## 🎯 Theoretical Basis
Vanilla CoT SFT teaches the model *how to reason*, but does **not** teach it *what not to do*.
Under surface perturbations, the model eagerly falls back on memorized shortcuts.

**Arm 2 (Contrastive SFT → DPO)** structures training data as contrastive preference pairs:
- **Chosen ($y^+$):** Reasoning trace that explicitly **rejects** the decoy template and derives the correct algorithm.
- **Rejected ($y^-$):** Memorized shortcut that fails on the perturbed task.

### Contrastive SFT Loss (from proposal §8.2):
$$\mathcal{L}_{\text{Contrastive}} = -\sum_{t=1}^{T} \log P_\theta(y_t^+ \mid x, y_{<t}^+) + \alpha \max \left( 0, \log P_\theta(y^- \mid x) - \log P_\theta(y^+ \mid x) + m \right)$$

### DPO Extension (Future Phase — Phase 2):
$$\mathcal{L}_{\text{DPO}} = -\mathbb{E} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y^+ | x)}{\pi_{\text{ref}}(y^+ | x)} - \beta \log \frac{\pi_\theta(y^- | x)}{\pi_{\text{ref}}(y^- | x)} \right) \right]$$

> **Where we are now:** We have `data/distillation/sft_contrastive_pairs.jsonl` (500 pairs) ready.
> Full DPO training with external data (SuperCorrect-style) is planned for **Phase 2** of the project.

In [ ]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("."))

from src.core.config import get_settings
from src.stage3_distillation.contrastive_builder import ContrastiveDatasetBuilder

settings = get_settings()
print("✅ Arm 2 (Contrastive SFT) Environment Initialized.")
print(f"   Contrastive file  : {settings.distillation.contrastive_file}")
print(f"   Distillation dir  : {settings.storage.distillation_cache_dir}")

---
## 1. Contrastive Corpus Inspection & Pair Quality

Each record must contain three required fields:
1. `prompt` — the perturbed problem statement.
2. `decoy_template` — the shortcut to reject ($y^-$).
3. `response` — the contrastive reasoning trace ($y^+$).

In [ ]:
contrastive_path = os.path.join(settings.storage.distillation_cache_dir, settings.distillation.contrastive_file)
records = ContrastiveDatasetBuilder.load_jsonl(contrastive_path)
print(f"Total contrastive pairs: {len(records):,}")

stats = ContrastiveDatasetBuilder.get_stats(contrastive_path)
print(f"Average token count   : {stats['avg_tokens']:,}")
print(f"Level breakdown       : {stats['level_breakdown']}")

sample = records[0]
print("\n--- SAMPLE RECORD ---")
print("task_id      :", sample['task_id'])
print("ladder_level :", sample.get('ladder_level', 'N/A'))
print("l0_task_id   :", sample.get('l0_task_id', 'N/A'))
print("\nPROMPT:\n", sample['prompt'][:300], "...")
print("\nDECOY TEMPLATE (to REJECT, y-):\n", sample.get('decoy_template', '')[:200])
print("\nCONTRASTIVE RESPONSE (y+):\n", sample['response'][:400], "...")

---
## 2. Level Distribution Visualization

In [ ]:
level_counts = stats['level_breakdown']
levels = sorted(level_counts.keys())
counts = [level_counts[l] for l in levels]

plt.figure(figsize=(7, 4), dpi=140)
bars = plt.bar(levels, counts, color='#FF6400', edgecolor='black', linewidth=0.5)
for bar, cnt in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(cnt), ha='center', va='bottom', fontsize=9)
plt.title('Contrastive Pairs — Distribution Across Ladder Levels', fontsize=11, fontweight='bold')
plt.xlabel('Ladder Level', fontsize=10)
plt.ylabel('Number of Pairs', fontsize=10)
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/arm2_contrastive_level_dist.png', dpi=140)
plt.show()
os.makedirs('results', exist_ok=True)
plt.savefig('results/arm2_contrastive_level_dist.png', dpi=140)


---
## 3. DPO Triplet Format Converter

For **Phase 2**: converts our contrastive corpus to TRL `DPOTrainer` `(prompt, chosen, rejected)` triplet format.

In [ ]:
def to_dpo_triplet(rec: dict) -> dict:
    """Converts a contrastive record to a HuggingFace DPO preference pair."""
    return {
        "prompt":   rec["prompt"],
        "chosen":   rec["response"],         # y+ : contrastive reasoning trace
        "rejected": rec.get("decoy_template", ""),  # y- : memorized shortcut
    }

dpo_sample = to_dpo_triplet(records[0])
print("✅ DPO Triplet Keys:", list(dpo_sample.keys()))
print(f"\n[prompt]   (first 100 chars): {dpo_sample['prompt'][:100]}")
print(f"[chosen]   (first 100 chars): {dpo_sample['chosen'][:100]}")
print(f"[rejected] (first 100 chars): {dpo_sample['rejected'][:100]}")
print("\n✅ Phase 2 DPO training corpus format verified — ready for TRL DPOTrainer!")